In [141]:
import numpy as np
import plotly.graph_objects as go
from numba import njit,prange

### Inversion Method

<div style="text-align: justify;text-justify: inter-word; font-size: 18px;">

The inversion method is a technique used to generate samples for a random variable *$X$* that follows a desired probability density function (PDF) *$f(X)$*. Suppose *$U$* is a random variable with a uniform distribution on the interval $[0,1]$, i.e., *$U \sim Uniform(0,1)$*. Samples of *$X$* can then be obtained by applying the inverse cumulative distribution function *$F(x)$* to *$U$*, namely *$X = F^{-1}(U)$*. </br>

In our 2D example, the desired PDF along x-axis is *$f(x)=πsin(2πx)$*, where $0 \leq x \leq 0.5$ and *π* is the normalization constant. Applying the inversion method, the position along the x-axis is given by *$X = \frac{cos^{-1}(1-2U)}{2π}$*. Along the y-axis, the position follows a simple Uniform distribution on the interval [0.1, 0.9].</br>

This distribution was selected since this PDF is not directly available as a predefined sampling distribution in either *numpy.random* or *scipy.stats*

</div>

In [142]:
def U(): return np.random.uniform()
ns=30
x_sources=[np.arccos(1-2*U())/(2*np.pi) for _ in range(ns)]
y_sources=np.random.uniform(0.1,0.9,ns)

## Discretization (FVM/FDM)

<div style="text-align: justify;text-justify: inter-word; font-size: 18px;">

$\frac{\partial{u_p}}{\partial{t}}$ = $a\left ( { \frac{u_E -2 u_P + u_W}{Δx^2}} \right)$ + $a\left ( { \frac{u_N -2 u_P + u_S}{Δy^2}} \right)$ - $U \left({\frac{u_P - u_W}{Δx}}\right) + Q_P$ </br>

After transforming to Cartesian indexing, the above expression yields a finite difference scheme (FDM), where the diffusion term is approximated using a central difference scheme (CDS) and the advection term using a backward difference scheme.</br>

For the time discretization, an explicit time-stepping scheme is used: $ u_{i,j}^{n+1} = u_{i,j}^{n} + r\left({u_{i+1,j}^n - 2u_{i,j}^n + u_{i-1,j}^n}\right) + r\left({u_{i,j+1}^n - 2u_{i,j}^n + u_{i,j-1}^n}\right) - c\left({u_{i,j}^n - u_{i-1,j}^n}\right) + Δt Q_{i,j}^n$ </br>

where $i,j=0,1,...,N$ and $Δx=Δy, \:$ leading to a single diffusion parameter  $r=a\frac{Δt}{Δx^2}$ and $c = U \frac{Δt}{Δx}.$

#### *Stability conditions*
$CFL: c = U \frac{Δt}{Δx}\leq 1$ and $r=a\frac{Δt}{Δx^2} \leq 0.25$.</br>

These conditions can be found, for example, in *Numerical Solution of Partial Differential Equations: Finite Difference Methods by G. D. Smith (3rd edition)*.

#### *Boundary conditions*

On the top, bottom and right boundaries, Neumann Boundary Conditions are imposed: $\left({\frac{\partial{u}}{\partial{x}}}\right)_s =0$, where $s=e,n,s \:$ </br>

 while at the left boundary the themperature is fixed at zero (Dirichlet condition) $u_w = 0.$ </br>

Linear interpolation is applied at all boundaries, and ghost points are introduced. For the Neumann boundary conditions, the resulting expressions are equivalent to those obtained by directly applying the finite difference method (FDM):

$u_{i,j-1}=u_{i,j},\; u_{i,j+1}=u_{i,j},\; u_{i+1,j}=u_{i,j}\;$</br>

which yield three individual equations for the three sides, with the ghost points eliminated. Similarly, two additional equations are derived for the upper-right and bottom-right corners using:
$u_{i+1,j-1}=u_{i,j}, \;u_{i+1,j+1}=u_{i,j}.$ </br>

For the Dirichlet boundary condition, linear interpolation leads to: $u_{i-1,j}=-u_{i,j}$ which is used to eliminate the ghost points in the discretized equation along the left boundary. Consequently, two additional equations are obtained for the upper-left and bottom-left corners
</div>

In [143]:
# --- Set up physical and numerical parameters ---

L = 1.0          # Length of the square computational domain [0,L] x [0,L]
T = 5.0          # Final simulation time

a = 0.005        # Diffusion coefficient
dx = dy = 0.01   # Spatial step sizes in x and y directions
dt = 0.0005      # Time step

U = 0.7          # Constant advection velocity in the x-direction

# Dimensionless advection and diffusion numbers
c = U * dt / dx
r = (a * dt) / (dx**2)

# Computational grids
x = np.arange(0, L + dx, dx)
y = np.arange(0, L + dx, dx)
t = np.arange(0, T + dt, dt)

# --- Temperature fields ---

# u_old stores the temperature field at the current time step
u_old = np.zeros((len(y), len(x)), dtype=np.float32)

# u_new stores the updated temperature field at the next time step
u_new = np.zeros_like(u_old)

# --- Saving field ---

save_every = 2   # Store one frame every 2 time steps
n_save = (len(t) - 1) // save_every + 1
u_saved = np.zeros((n_save, len(y), len(x)), dtype=np.float32)

u_saved[0, :, :] = u_old.copy()     # Store initial condition

### Source Fields
<div style="text-align: justify; text-justify: inter-word; font-size: 18px;">

The source activation is modeled as a discrete-time approximation of a Poisson process. At each time step, every inactive source has probability $\lambda \Delta t$ of becoming active, where $\lambda$ is the firing rate. Therefore, the number of activations over a fixed time interval follows a Poisson distribution, while the waiting times between activations are exponentially distributed.

</div>

In [144]:
# active[k] indicates whether source k is currently injecting

active = np.zeros(ns, dtype=np.int32)   # 0 = inactive, 1 = active

# remaining[k] stores how much injection time is left for source k
remaining = np.zeros(ns, dtype=np.float32)

# Total source term at the current time step
source = np.zeros_like(u_old)

rate = 0.5    # Poisson firing rate

# q[k,j,i] stores the spatial Gaussian profile of source k
q = np.zeros((ns, len(y), len(x)), dtype=np.float32)

for k in range(ns):
    for i in range(len(x)):
        for j in range(len(y)):

            # Squared distance from source k in x and y directions
            argx = ((x[i] - x_sources[k])**2) / 0.005**2
            argy = ((y[j] - y_sources[k])**2) / 0.005**2

            # Gaussian source profile.
            # min(...,50) prevents numerical underflow/overflow in the exponential.
            q[k, j, i] = 1000 * np.exp( -min(argx, 50.0) - min(argy, 50.0) ) # min(...,50) prevents numerical underflow/overflow in the exponential.

In [145]:
@njit(parallel=True)
def FTCS(dt, ns, u_old, u_new, u_history, source, c, r, nt, ny, nx, save_every, active, remaining, rate, q):

    # Duration of each source injection once it becomes active
    injection_time = 0.15

    # Index used for storing snapshots in u_history
    save_idx = 1

    # Stability check:
    if (c < 1) and (r < 0.25):

        for n in range(0, nt):

            # Reset source field at each time step
            source[:, :] = 0.0

            # --- Poisson source activation ---
            # Each source independently fires with probability rate*dt.
            # Once active, it contributes for injection_time seconds.
            for k in range(ns):

                if active[k] == 0:
                    if np.random.rand() < rate * dt:
                        active[k] = 1
                        remaining[k] = injection_time

                if active[k] == 1:
                    source += q[k]
                    remaining[k] -= dt

                    if remaining[k] <= 0.0:
                        active[k] = 0

                        # --- Interior ---
            for j in prange(1,ny-1):
                for i in range(1,nx-1):
                    u_new[j,i]=u_old[j,i]+r*(u_old[j,i+1]-2.0*u_old[j,i]+u_old[j,i-1])+r*(u_old[j+1,i]-2.0*u_old[j,i]+u_old[j-1,i])-c*(u_old[j,i]-u_old[j,i-1])+dt*source[j,i]

            # --- Top and Bottom Boundaries ---
            for i in range(1,nx-1):
                u_new[ny-1,i]=u_old[ny-1,i]+r*(u_old[ny-1,i+1]-2.0*u_old[ny-1,i]+u_old[ny-1,i-1])+r*(-1.0*u_old[ny-1,i]+u_old[ny-2,i])-c*(u_old[ny-1,i]-u_old[ny-1,i-1]) # (y=L)
                u_new[0,i]=u_old[0,i]+r*(u_old[0,i+1]-2.0*u_old[0,i]+u_old[0,i-1])+r*(u_old[1,i]-1.0*u_old[0,i])-c*(u_old[0,i]-u_old[0,i-1]) # (y=0)

            # --- Left and Right Boundaries ---
            for j in range(1,ny-1):
                u_new[j,nx-1]=u_old[j,nx-1]+r*(-1.0*u_old[j,nx-1]+u_old[j,nx-2])+r*(u_old[j+1,nx-1]-2.0*u_old[j,nx-1]+u_old[j-1,nx-1])-c*(u_old[j,nx-1]-u_old[j,nx-2]) # (x=L)
                u_new[j,0]=u_old[j,0]+r*(u_old[j,1]-3.0*u_old[j,0])+r*(u_old[j+1,0]-2.0*u_old[j,0]+u_old[j-1,0])-c*(2.0*u_old[j,0]) # (x=0)

            # --- Four Corners ---
            u_new[0,0]=u_old[0,0]+r*(u_old[0,1]-3.0*u_old[0,0])+r*(u_old[1,0]-3.0*u_old[0,0])-c*(2.0*u_old[0,0]) # Bottom-left corner
            u_new[ny-1,0]=u_old[ny-1,0]+r*(u_old[ny-1,1]-3.0*u_old[ny-1,0])+r*(-3.0*u_old[ny-1,0]+u_old[ny-2,0])-c*(2.0*u_old[ny-1,0]) # Top=left corner
            u_new[0,nx-1]=u_old[0,nx-1]+r*(-1.0*u_old[0,nx-1]+u_old[0,nx-2])+r*(u_old[1,nx-1]-1.0*u_old[0,nx-1])-c*(u_old[0,nx-1]-u_old[0,nx-2]) # Bottom-right corner
            u_new[ny-1,nx-1]=u_old[ny-1,nx-1]+r*(-1.0*u_old[ny-1,nx-1]+u_old[ny-1,nx-2])+r*(-1.0*u_old[ny-1,nx-1]+u_old[ny-2,nx-1])-c*(u_old[ny-1,nx-1]-u_old[ny-1,nx-2]) # Upper-right corner
            #u_new[:, 0] = 0.0 # Left side (x=0) using FDM

            # --- Update ---
            u_old,u_new=u_new,u_old

            # --- Save ---
            if (n + 1) % save_every == 0:
                u_history[save_idx,:,:] = np.copy(u_old)
                save_idx += 1
    else: print("Unstable System")
    return u_history

u_final=FTCS(dt,ns,u_old,u_new,u_saved,source,c,r,len(t),len(y),len(x),save_every,active,remaining,rate,q)
#FTCS.parallel_diagnostics(level=2)

# Animation (Plotly)

In [ ]:
frames=[]
for k in range(1,len(u_final[:,0,0]),5):
    f_plot=u_final[k].copy()
    frames.append(go.Frame(data=[go.Heatmap(x=x,y=y,z=f_plot,zsmooth="best",colorscale="Jet",zmin=0,zmax=0.5,colorbar=dict(yanchor="middle",len=425,lenmode="pixels"))]))
f0=u_final[0,:,:].copy()
fig=go.Figure(data=[go.Heatmap(x=x,y=y,z=f0,colorscale="Jet",zsmooth="best",zmin=0,zmax=0.5,colorbar=dict(yanchor="middle",len=425,lenmode="pixels"))],
    layout=go.Layout(updatemenus=[dict(x=0.1,y=-0.12,xanchor="center",direction="left",type="buttons", showactive=False,
        buttons=[dict(label="Play", method="animate",args=[None, dict(frame=dict(duration=50,redraw=True),transition=dict(duration=0),fromcurrent=True,mode="immediate")]),
                 dict(label="Pause",method="animate",args=[[None], dict(frame=dict(duration=50,redraw=False),transition=dict(duration=0),mode="immediate")])])]),frames=frames)

fig.update_layout(title=dict(text="Heat Sources Temperature",x=0.5,y=0.925,
                             font=dict(family="Linux Libertine O",color="Black",size=30)),
                             width=800,height=600,plot_bgcolor="White",
                             xaxis_title="x",xaxis_title_font=dict(color="black",size=30,family="Linux Libertine O"),
                             yaxis_title="y",yaxis_title_font=dict(color="black",size=30,family="Linux Libertine O"))
fig.update_xaxes(range=[0,1],linecolor="black",showline=True,mirror=True,tickfont=dict(size=20,color="black"), ticks="outside", ticklen=8, tickwidth=2)
fig.update_yaxes(range=[0,1],linecolor="black",showline=True,mirror=True,tickfont=dict(size=20,color="black"), ticks="outside", ticklen=8, tickwidth=2)
fig.show(auto_play=False)
fig.write_html("Animation.html",auto_play=False,include_plotlyjs="cdn")